# 21 — FDR Correction for Susceptibility Results

This notebook adds Benjamini–Hochberg FDR correction to the susceptibility analyses.

It corrects p-values from:

1. Multimodel raw/ERP/power susceptibility correlations
2. ICA multimodel susceptibility correlations
3. Exploratory susceptibility meta-model prediction correlations

The main outputs are corrected q-values and tables of effects that survive FDR correction.

Primary interpretation should focus on Spearman correlations because most susceptibility analyses are rank-based and robust to nonlinearity/outliers.

Important:
- This notebook does not rerun EEG preprocessing, ICA, classifiers, or neural models.
- It only reads existing result tables and adds q-values.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oddball_eeg.io import save_dataframe
from oddball_eeg.stats import benjamini_hochberg
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

set_pandas_display_options()

# ------------------------------------------------------------
# Input result folders
# ------------------------------------------------------------

RAW_TABLE_DIR = Path("results") / "thesis_artifacts" / "multimodel_classification_susceptibility" / "tables"

ICA_TABLE_DIR = Path("results") / "thesis_artifacts" / "ica_multimodel_classification_susceptibility" / "tables"

META_TABLE_DIR = Path("results") / "thesis_artifacts" / "susceptibility_meta_model" / "tables"

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUT_ROOT = Path("results") / "thesis_artifacts" / "fdr_correction_susceptibility"
TABLE_DIR = OUT_ROOT / "tables"
FIG_DIR = OUT_ROOT / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

Q_THRESHOLDS = [0.05, 0.10]

print("Raw/ERP/power input:", RAW_TABLE_DIR)
print("ICA input:", ICA_TABLE_DIR)
print("Meta-model input:", META_TABLE_DIR)
print("Output root:", OUT_ROOT)

In [ ]:
def add_fdr_by_family(
        df: pd.DataFrame,
        *,
        p_col: str,
        family_cols: list[str],
        q_col: str,
        sig_prefix: str,
) -> pd.DataFrame:
    """
    Add BH-FDR q-values within each specified family.

    Example family:
        model
        model + dataset
        target_mode + classification_model
    """
    out = df.copy()

    out[q_col] = np.nan

    for _, idx in out.groupby(family_cols, dropna=False).groups.items():
        idx = list(idx)
        out.loc[idx, q_col] = benjamini_hochberg(out.loc[idx, p_col].values)

    for q_thr in Q_THRESHOLDS:
        label = str(q_thr).replace(".", "")
        out[f"{sig_prefix}_fdr_{label}"] = out[q_col] <= q_thr

    return out


def add_standard_fdr_columns(
        df: pd.DataFrame,
        *,
        family_cols: list[str],
        pearson_p_col: str,
        spearman_p_col: str,
        pearson_q_col: str,
        spearman_q_col: str,
) -> pd.DataFrame:
    """
    Add Pearson and Spearman FDR correction columns.
    """
    out = df.copy()

    if pearson_p_col in out.columns:
        out = add_fdr_by_family(
            out,
            p_col=pearson_p_col,
            family_cols=family_cols,
            q_col=pearson_q_col,
            sig_prefix=pearson_q_col,
        )

    if spearman_p_col in out.columns:
        out = add_fdr_by_family(
            out,
            p_col=spearman_p_col,
            family_cols=family_cols,
            q_col=spearman_q_col,
            sig_prefix=spearman_q_col,
        )

    return out


def summarize_fdr_table(
        df: pd.DataFrame,
        *,
        table_name: str,
        family_cols: list[str],
        q_col: str,
        p_col: str,
        effect_col: str,
) -> pd.DataFrame:
    """
    Summarize number of FDR-significant effects by family.
    """
    rows = []

    for keys, g in df.groupby(family_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "table_name": table_name,
            **dict(zip(family_cols, keys)),
            "n_tests": int(np.isfinite(g[p_col]).sum()),
            "n_fdr_05": int((g[q_col] <= 0.05).sum()),
            "n_fdr_10": int((g[q_col] <= 0.10).sum()),
            "min_p": float(np.nanmin(g[p_col])) if np.isfinite(g[p_col]).any() else np.nan,
            "min_q": float(np.nanmin(g[q_col])) if np.isfinite(g[q_col]).any() else np.nan,
            "max_abs_effect": float(np.nanmax(np.abs(g[effect_col]))) if effect_col in g.columns else np.nan,
        }

        rows.append(row)

    return pd.DataFrame(rows)


def top_fdr_significant(
        df: pd.DataFrame,
        *,
        q_col: str,
        effect_col: str,
        q_threshold: float = 0.05,
        n_top: int = 50,
) -> pd.DataFrame:
    """
    Return top FDR-significant rows sorted by q-value and absolute effect.
    """
    out = df[df[q_col] <= q_threshold].copy()

    if len(out) == 0:
        return out

    out["abs_effect_for_sort"] = np.abs(out[effect_col])

    out = out.sort_values(
        [q_col, "abs_effect_for_sort"],
        ascending=[True, False],
    ).head(n_top)

    return out.drop(columns=["abs_effect_for_sort"], errors="ignore")

In [ ]:
raw_within_path = RAW_TABLE_DIR / "within_dataset_correlations_by_model.csv"
raw_grand_path = RAW_TABLE_DIR / "zscored_grand_correlations_by_model.csv"
raw_grand_report_path = RAW_TABLE_DIR / "zscored_grand_correlations_report_filtered_by_model.csv"
raw_general_path = RAW_TABLE_DIR / "general_classifiability_correlations.csv"

for p in [
    raw_within_path,
    raw_grand_path,
    raw_grand_report_path,
    raw_general_path,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required raw/ERP/power table:\n{p}")

raw_within_df = pd.read_csv(raw_within_path)
raw_grand_df = pd.read_csv(raw_grand_path)
raw_grand_report_df = pd.read_csv(raw_grand_report_path)
raw_general_df = pd.read_csv(raw_general_path)

print("Raw within rows:", len(raw_within_df))
print("Raw grand rows:", len(raw_grand_df))
print("Raw grand report rows:", len(raw_grand_report_df))
print("Raw general classifiability rows:", len(raw_general_df))

display(round_numeric_columns(raw_grand_report_df.head(), 4))

In [ ]:
# Within-dataset raw/ERP/power correlations:
# FDR family = model × dataset.
raw_within_fdr_df = add_standard_fdr_columns(
    raw_within_df,
    family_cols=["model", "dataset_label"],
    pearson_p_col="pearson_p",
    spearman_p_col="spearman_p",
    pearson_q_col="pearson_q",
    spearman_q_col="spearman_q",
)

# Grand z-scored raw/ERP/power correlations:
# FDR family = model.
raw_grand_fdr_df = add_standard_fdr_columns(
    raw_grand_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

# Primary report-filtered grand correlations:
# Same FDR family = model, but only among reportable features.
raw_grand_report_fdr_df = add_standard_fdr_columns(
    raw_grand_report_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

# General mean-AUC classifiability:
# One family across reportable features.
raw_general_fdr_df = add_standard_fdr_columns(
    raw_general_df,
    family_cols=["target"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

save_dataframe(raw_within_fdr_df, TABLE_DIR / "raw_within_dataset_correlations_by_model_fdr.csv")
save_dataframe(raw_grand_fdr_df, TABLE_DIR / "raw_zscored_grand_correlations_by_model_fdr.csv")
save_dataframe(raw_grand_report_fdr_df, TABLE_DIR / "raw_zscored_grand_correlations_report_filtered_by_model_fdr.csv")
save_dataframe(raw_general_fdr_df, TABLE_DIR / "raw_general_classifiability_correlations_fdr.csv")

print("Raw/ERP/power FDR correction complete.")

display(
    round_numeric_columns(
        raw_grand_report_fdr_df
        .sort_values(["model", "spearman_q_z", "abs_spearman_rho_z"])
        .groupby("model")
        .head(10),
        4,
    )
)

In [ ]:
ica_within_path = ICA_TABLE_DIR / "ica_within_dataset_correlations_by_model.csv"
ica_grand_path = ICA_TABLE_DIR / "ica_zscored_grand_correlations_by_model.csv"
ica_grand_report_path = ICA_TABLE_DIR / "ica_zscored_grand_correlations_report_filtered_by_model.csv"

for p in [
    ica_within_path,
    ica_grand_path,
    ica_grand_report_path,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required ICA table:\n{p}")

ica_within_df = pd.read_csv(ica_within_path)
ica_grand_df = pd.read_csv(ica_grand_path)
ica_grand_report_df = pd.read_csv(ica_grand_report_path)

print("ICA within rows:", len(ica_within_df))
print("ICA grand rows:", len(ica_grand_df))
print("ICA grand report rows:", len(ica_grand_report_df))

display(round_numeric_columns(ica_grand_report_df.head(), 4))

In [ ]:
# Within-dataset ICA correlations:
# FDR family = model × dataset.
ica_within_fdr_df = add_standard_fdr_columns(
    ica_within_df,
    family_cols=["model", "dataset_label"],
    pearson_p_col="pearson_p",
    spearman_p_col="spearman_p",
    pearson_q_col="pearson_q",
    spearman_q_col="spearman_q",
)

# Full grand ICA correlations:
# FDR family = model.
ica_grand_fdr_df = add_standard_fdr_columns(
    ica_grand_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

# Primary report-filtered grand ICA correlations:
# Same FDR family = model.
ica_grand_report_fdr_df = add_standard_fdr_columns(
    ica_grand_report_df,
    family_cols=["model"],
    pearson_p_col="pearson_p_z",
    spearman_p_col="spearman_p_z",
    pearson_q_col="pearson_q_z",
    spearman_q_col="spearman_q_z",
)

save_dataframe(ica_within_fdr_df, TABLE_DIR / "ica_within_dataset_correlations_by_model_fdr.csv")
save_dataframe(ica_grand_fdr_df, TABLE_DIR / "ica_zscored_grand_correlations_by_model_fdr.csv")
save_dataframe(ica_grand_report_fdr_df, TABLE_DIR / "ica_zscored_grand_correlations_report_filtered_by_model_fdr.csv")

print("ICA FDR correction complete.")

display(
    round_numeric_columns(
        ica_grand_report_fdr_df
        .sort_values(["model", "spearman_q_z", "abs_spearman_rho_z"])
        .groupby("model")
        .head(10),
        4,
    )
)

In [ ]:
meta_summary_path = META_TABLE_DIR / "all_lodo_summary_metrics.csv"

if not meta_summary_path.exists():
    print("Meta-model summary table not found. Skipping meta-model FDR.")
    meta_fdr_df = pd.DataFrame()
else:
    meta_df = pd.read_csv(meta_summary_path)

    # FDR family:
    # target_mode × classification_model
    # This asks: within each prediction target family, which meta-model correlations survive correction?
    meta_fdr_df = add_standard_fdr_columns(
        meta_df,
        family_cols=["target_mode", "classification_model"],
        pearson_p_col="pearson_p",
        spearman_p_col="spearman_p",
        pearson_q_col="pearson_q",
        spearman_q_col="spearman_q",
    )

    save_dataframe(
        meta_fdr_df,
        TABLE_DIR / "meta_model_lodo_summary_metrics_fdr.csv",
    )

    print("Meta-model FDR correction complete.")
    display(
        round_numeric_columns(
            meta_fdr_df
            .sort_values(["target_mode", "classification_model", "spearman_q", "spearman_rho"])
            .groupby(["target_mode", "classification_model"])
            .head(10),
            4,
        )
    )

In [ ]:
summary_frames = []

summary_frames.append(
    summarize_fdr_table(
        raw_within_fdr_df,
        table_name="raw_within_dataset",
        family_cols=["model", "dataset_label"],
        q_col="spearman_q",
        p_col="spearman_p",
        effect_col="spearman_rho",
    )
)

summary_frames.append(
    summarize_fdr_table(
        raw_grand_report_fdr_df,
        table_name="raw_grand_report_filtered",
        family_cols=["model"],
        q_col="spearman_q_z",
        p_col="spearman_p_z",
        effect_col="spearman_rho_z",
    )
)

summary_frames.append(
    summarize_fdr_table(
        raw_general_fdr_df,
        table_name="raw_general_classifiability",
        family_cols=["target"],
        q_col="spearman_q_z",
        p_col="spearman_p_z",
        effect_col="spearman_rho_z",
    )
)

summary_frames.append(
    summarize_fdr_table(
        ica_within_fdr_df,
        table_name="ica_within_dataset",
        family_cols=["model", "dataset_label"],
        q_col="spearman_q",
        p_col="spearman_p",
        effect_col="spearman_rho",
    )
)

summary_frames.append(
    summarize_fdr_table(
        ica_grand_report_fdr_df,
        table_name="ica_grand_report_filtered",
        family_cols=["model"],
        q_col="spearman_q_z",
        p_col="spearman_p_z",
        effect_col="spearman_rho_z",
    )
)

if len(meta_fdr_df) > 0:
    summary_frames.append(
        summarize_fdr_table(
            meta_fdr_df,
            table_name="meta_model_lodo",
            family_cols=["target_mode", "classification_model"],
            q_col="spearman_q",
            p_col="spearman_p",
            effect_col="spearman_rho",
        )
    )

fdr_summary_df = pd.concat(summary_frames, ignore_index=True)

save_dataframe(
    fdr_summary_df,
    TABLE_DIR / "fdr_summary_by_family.csv",
)

display(round_numeric_columns(fdr_summary_df, 4))

In [ ]:
significant_tables = []

# Raw grand report-filtered
sig = top_fdr_significant(
    raw_grand_report_fdr_df,
    q_col="spearman_q_z",
    effect_col="spearman_rho_z",
    q_threshold=0.05,
)
if len(sig) > 0:
    sig = sig.copy()
    sig["analysis"] = "raw_grand_report_filtered"
    sig["q_col_used"] = "spearman_q_z"
    sig["effect_col_used"] = "spearman_rho_z"
    significant_tables.append(sig)

# Raw within-dataset
sig = top_fdr_significant(
    raw_within_fdr_df,
    q_col="spearman_q",
    effect_col="spearman_rho",
    q_threshold=0.05,
)
if len(sig) > 0:
    sig = sig.copy()
    sig["analysis"] = "raw_within_dataset"
    sig["q_col_used"] = "spearman_q"
    sig["effect_col_used"] = "spearman_rho"
    significant_tables.append(sig)

# Raw general classifiability
sig = top_fdr_significant(
    raw_general_fdr_df,
    q_col="spearman_q_z",
    effect_col="spearman_rho_z",
    q_threshold=0.05,
)
if len(sig) > 0:
    sig = sig.copy()
    sig["analysis"] = "raw_general_classifiability"
    sig["q_col_used"] = "spearman_q_z"
    sig["effect_col_used"] = "spearman_rho_z"
    significant_tables.append(sig)

# ICA grand report-filtered
sig = top_fdr_significant(
    ica_grand_report_fdr_df,
    q_col="spearman_q_z",
    effect_col="spearman_rho_z",
    q_threshold=0.05,
)
if len(sig) > 0:
    sig = sig.copy()
    sig["analysis"] = "ica_grand_report_filtered"
    sig["q_col_used"] = "spearman_q_z"
    sig["effect_col_used"] = "spearman_rho_z"
    significant_tables.append(sig)

# ICA within-dataset
sig = top_fdr_significant(
    ica_within_fdr_df,
    q_col="spearman_q",
    effect_col="spearman_rho",
    q_threshold=0.05,
)
if len(sig) > 0:
    sig = sig.copy()
    sig["analysis"] = "ica_within_dataset"
    sig["q_col_used"] = "spearman_q"
    sig["effect_col_used"] = "spearman_rho"
    significant_tables.append(sig)

# Meta-model
if len(meta_fdr_df) > 0:
    sig = top_fdr_significant(
        meta_fdr_df,
        q_col="spearman_q",
        effect_col="spearman_rho",
        q_threshold=0.05,
    )
    if len(sig) > 0:
        sig = sig.copy()
        sig["analysis"] = "meta_model_lodo"
        sig["q_col_used"] = "spearman_q"
        sig["effect_col_used"] = "spearman_rho"
        significant_tables.append(sig)

if len(significant_tables) > 0:
    all_sig_fdr05_df = pd.concat(significant_tables, ignore_index=True)
else:
    all_sig_fdr05_df = pd.DataFrame()

save_dataframe(
    all_sig_fdr05_df,
    TABLE_DIR / "all_spearman_fdr05_significant_results.csv",
)

print("Number of Spearman FDR q <= 0.05 results:", len(all_sig_fdr05_df))

if len(all_sig_fdr05_df) > 0:
    display(round_numeric_columns(all_sig_fdr05_df.head(80), 4))
else:
    print("No Spearman effects survived FDR q <= 0.05.")

In [ ]:
exploratory_tables = []

def append_exploratory(df, analysis_name, q_col, effect_col):
    sig = top_fdr_significant(
        df,
        q_col=q_col,
        effect_col=effect_col,
        q_threshold=0.10,
        n_top=100,
    )
    if len(sig) > 0:
        sig = sig.copy()
        sig["analysis"] = analysis_name
        sig["q_col_used"] = q_col
        sig["effect_col_used"] = effect_col
        exploratory_tables.append(sig)

append_exploratory(raw_grand_report_fdr_df, "raw_grand_report_filtered", "spearman_q_z", "spearman_rho_z")
append_exploratory(raw_within_fdr_df, "raw_within_dataset", "spearman_q", "spearman_rho")
append_exploratory(raw_general_fdr_df, "raw_general_classifiability", "spearman_q_z", "spearman_rho_z")
append_exploratory(ica_grand_report_fdr_df, "ica_grand_report_filtered", "spearman_q_z", "spearman_rho_z")
append_exploratory(ica_within_fdr_df, "ica_within_dataset", "spearman_q", "spearman_rho")

if len(meta_fdr_df) > 0:
    append_exploratory(meta_fdr_df, "meta_model_lodo", "spearman_q", "spearman_rho")

if len(exploratory_tables) > 0:
    all_sig_fdr10_df = pd.concat(exploratory_tables, ignore_index=True)
else:
    all_sig_fdr10_df = pd.DataFrame()

save_dataframe(
    all_sig_fdr10_df,
    TABLE_DIR / "all_spearman_fdr10_exploratory_results.csv",
)

print("Number of Spearman FDR q <= 0.10 results:", len(all_sig_fdr10_df))

if len(all_sig_fdr10_df) > 0:
    display(round_numeric_columns(all_sig_fdr10_df.head(100), 4))
else:
    print("No Spearman effects survived FDR q <= 0.10.")

In [ ]:
exploratory_tables = []

def append_exploratory(df, analysis_name, q_col, effect_col):
    sig = top_fdr_significant(
        df,
        q_col=q_col,
        effect_col=effect_col,
        q_threshold=0.10,
        n_top=100,
    )
    if len(sig) > 0:
        sig = sig.copy()
        sig["analysis"] = analysis_name
        sig["q_col_used"] = q_col
        sig["effect_col_used"] = effect_col
        exploratory_tables.append(sig)

append_exploratory(raw_grand_report_fdr_df, "raw_grand_report_filtered", "spearman_q_z", "spearman_rho_z")
append_exploratory(raw_within_fdr_df, "raw_within_dataset", "spearman_q", "spearman_rho")
append_exploratory(raw_general_fdr_df, "raw_general_classifiability", "spearman_q_z", "spearman_rho_z")
append_exploratory(ica_grand_report_fdr_df, "ica_grand_report_filtered", "spearman_q_z", "spearman_rho_z")
append_exploratory(ica_within_fdr_df, "ica_within_dataset", "spearman_q", "spearman_rho")

if len(meta_fdr_df) > 0:
    append_exploratory(meta_fdr_df, "meta_model_lodo", "spearman_q", "spearman_rho")

if len(exploratory_tables) > 0:
    all_sig_fdr10_df = pd.concat(exploratory_tables, ignore_index=True)
else:
    all_sig_fdr10_df = pd.DataFrame()

save_dataframe(
    all_sig_fdr10_df,
    TABLE_DIR / "all_spearman_fdr10_exploratory_results.csv",
)

print("Number of Spearman FDR q <= 0.10 results:", len(all_sig_fdr10_df))

if len(all_sig_fdr10_df) > 0:
    display(round_numeric_columns(all_sig_fdr10_df.head(100), 4))
else:
    print("No Spearman effects survived FDR q <= 0.10.")

In [ ]:
# Compact report table for raw grand correlations
raw_grand_report_compact = raw_grand_report_fdr_df[[
    "model",
    "model_label",
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "spearman_q_z",
    "spearman_q_z_fdr_005",
    "spearman_q_z_fdr_01",
    "pearson_r_z",
    "pearson_p_z",
    "pearson_q_z",
    "pearson_q_z_fdr_005",
    "pearson_q_z_fdr_01",
]].sort_values(["model", "spearman_q_z", "spearman_p_z"])

save_dataframe(
    raw_grand_report_compact,
    TABLE_DIR / "report_raw_grand_correlations_with_qvalues.csv",
)

# Compact report table for ICA grand correlations
ica_grand_report_compact = ica_grand_report_fdr_df[[
    "model",
    "model_label",
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "spearman_q_z",
    "spearman_q_z_fdr_005",
    "spearman_q_z_fdr_01",
    "pearson_r_z",
    "pearson_p_z",
    "pearson_q_z",
    "pearson_q_z_fdr_005",
    "pearson_q_z_fdr_01",
]].sort_values(["model", "spearman_q_z", "spearman_p_z"])

save_dataframe(
    ica_grand_report_compact,
    TABLE_DIR / "report_ica_grand_correlations_with_qvalues.csv",
)

# Compact report table for general raw classifiability
raw_general_compact = raw_general_fdr_df[[
    "target",
    "feature",
    "n",
    "n_valid_grand",
    "n_valid_datasets",
    "spearman_rho_z",
    "spearman_p_z",
    "spearman_q_z",
    "spearman_q_z_fdr_005",
    "spearman_q_z_fdr_01",
    "pearson_r_z",
    "pearson_p_z",
    "pearson_q_z",
    "pearson_q_z_fdr_005",
    "pearson_q_z_fdr_01",
]].sort_values(["target", "spearman_q_z", "spearman_p_z"])

save_dataframe(
    raw_general_compact,
    TABLE_DIR / "report_general_classifiability_correlations_with_qvalues.csv",
)

print("Main report tables saved.")

print("\nRaw grand report top rows:")
display(round_numeric_columns(raw_grand_report_compact.groupby("model").head(8), 4))

print("\nICA grand report top rows:")
display(round_numeric_columns(ica_grand_report_compact.groupby("model").head(8), 4))

print("\nGeneral classifiability top rows:")
display(round_numeric_columns(raw_general_compact.head(12), 4))

In [ ]:
plot_df = fdr_summary_df.copy()

# Make a readable family label.
label_cols = [
    c for c in [
        "table_name",
        "model",
        "dataset_label",
        "target",
        "target_mode",
        "classification_model",
    ]
    if c in plot_df.columns
]

plot_df["family_label"] = plot_df[label_cols].astype(str).agg(" | ".join, axis=1)

plot_df = plot_df.sort_values("n_fdr_05", ascending=True)

fig, ax = plt.subplots(figsize=(11, max(5, 0.28 * len(plot_df))))

ax.barh(plot_df["family_label"], plot_df["n_fdr_05"])

ax.set_xlabel("Number of Spearman effects with FDR q ≤ 0.05")
ax.set_title("FDR-significant susceptibility effects by family")
ax.grid(axis="x", alpha=0.3)

fig.tight_layout()
save_figure(fig, FIG_DIR / "fdr_significant_counts_by_family.png")
plt.show()

In [ ]:
for model, g in raw_grand_report_fdr_df.groupby("model"):
    g = g.copy()
    g = g.sort_values("spearman_q_z", ascending=True).head(20)
    g = g.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(g["feature"], g["spearman_rho_z"])
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel("Spearman rho with z-scored AUC")
    ax.set_title(f"Raw/ERP/power grand correlations with FDR q-values — {model}")
    ax.grid(axis="x", alpha=0.3)

    # Add q-value text
    for i, (_, row) in enumerate(g.iterrows()):
        ax.text(
            row["spearman_rho_z"],
            i,
            f" q={row['spearman_q_z']:.3f}",
            va="center",
            ha="left" if row["spearman_rho_z"] >= 0 else "right",
            fontsize=8,
        )

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"raw_grand_fdr_barplot_{model}.png")
    plt.show()

In [ ]:
for model, g in ica_grand_report_fdr_df.groupby("model"):
    g = g.copy()
    g = g.sort_values("spearman_q_z", ascending=True).head(20)
    g = g.sort_values("spearman_rho_z", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.barh(g["feature"], g["spearman_rho_z"])
    ax.axvline(0.0, linewidth=1)

    ax.set_xlabel("Spearman rho with z-scored AUC")
    ax.set_title(f"ICA grand correlations with FDR q-values — {model}")
    ax.grid(axis="x", alpha=0.3)

    for i, (_, row) in enumerate(g.iterrows()):
        ax.text(
            row["spearman_rho_z"],
            i,
            f" q={row['spearman_q_z']:.3f}",
            va="center",
            ha="left" if row["spearman_rho_z"] >= 0 else "right",
            fontsize=8,
        )

    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"ica_grand_fdr_barplot_{model}.png")
    plt.show()

In [ ]:
print("=" * 90)
print("FDR interpretation helper")
print("=" * 90)

print("\nPrimary raw/ERP/power grand correlations with Spearman FDR q <= 0.05:")
raw_sig = raw_grand_report_fdr_df[raw_grand_report_fdr_df["spearman_q_z"] <= 0.05]
if len(raw_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            raw_sig.sort_values(["model", "spearman_q_z"])[
                [
                    "model",
                    "feature",
                    "spearman_rho_z",
                    "spearman_p_z",
                    "spearman_q_z",
                    "n_valid_grand",
                    "n_valid_datasets",
                ]
            ],
            4,
        )
    )

print("\nPrimary ICA grand correlations with Spearman FDR q <= 0.05:")
ica_sig = ica_grand_report_fdr_df[ica_grand_report_fdr_df["spearman_q_z"] <= 0.05]
if len(ica_sig) == 0:
    print(" - None survived q <= 0.05.")
else:
    display(
        round_numeric_columns(
            ica_sig.sort_values(["model", "spearman_q_z"])[
                [
                    "model",
                    "feature",
                    "spearman_rho_z",
                    "spearman_p_z",
                    "spearman_q_z",
                    "n_valid_grand",
                    "n_valid_datasets",
                ]
            ],
            4,
        )
    )

print("\nMeta-model LODO results with Spearman FDR q <= 0.05:")
if len(meta_fdr_df) == 0:
    print(" - Meta-model FDR was skipped.")
else:
    meta_sig = meta_fdr_df[meta_fdr_df["spearman_q"] <= 0.05]
    if len(meta_sig) == 0:
        print(" - None survived q <= 0.05.")
    else:
        display(
            round_numeric_columns(
                meta_sig.sort_values(["target_mode", "classification_model", "spearman_q"])[
                    [
                        "target_mode",
                        "classification_model",
                        "feature_set",
                        "meta_model",
                        "spearman_rho",
                        "spearman_p",
                        "spearman_q",
                        "rmse",
                        "rmse_improvement_pct_vs_dummy",
                    ]
                ],
                4,
            )
        )

print("\nSuggested wording:")
print(
    "After Benjamini-Hochberg FDR correction, report only q-values from the "
    "appropriate model/family. If few or no effects survive q <= 0.05, describe "
    "the p < 0.05 findings as exploratory and emphasize the corrected q-values."
)

In [ ]:
print("Outputs saved under:", OUT_ROOT)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" -", p)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p)